In [1]:
# Cell 1: Verify GPU and install libraries
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Install compatible versions (same as sentiment notebook)
!pip uninstall -y transformers accelerate datasets -q
!pip install transformers==4.44.2 datasets==2.19.0 accelerate==0.33.0 scikit-learn==1.4.2 numpy==1.26.4 -q

print("\n✅ All libraries installed successfully!")
print("⚠️  Now go to Runtime -> Restart session, then run all cells again!")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB

✅ All libraries installed successfully!
⚠️  Now go to Runtime -> Restart session, then run all cells again!


In [2]:
# Cell 2: Create synthetic covenant breach training dataset
# SEC EDGAR manual labeling would take days, so we create a high-quality
# synthetic dataset that mimics real covenant language from 10-K filings.
# This is a legitimate approach used in production NLP when labeled data is scarce.
# The model learns the PATTERNS of breach vs compliance language.

import random
random.seed(42)

# Breach examples - language indicating covenant violations
breach_templates = [
    # Debt covenant breaches
    "The company failed to maintain the minimum debt service coverage ratio of 1.20x as required under Section {section}, with actual DSCR falling to {ratio}x.",
    "As of the reporting date, the borrower was in violation of the maximum leverage ratio covenant of {max_ratio}x, with actual leverage at {ratio}x.",
    "The company breached the interest coverage ratio requirement of {min_ratio}x specified in the credit agreement, reporting only {ratio}x.",
    "Notice of default was issued under Section {section} due to the borrower's failure to maintain the required current ratio of {min_ratio}x.",
    "The borrower has failed to comply with the financial covenant requiring a minimum net worth of ${amount} million as specified in Section {section}.",
    "Due to declining revenues, the company was unable to meet the minimum EBITDA threshold of ${amount} million required under the revolving credit facility.",
    "The issuer triggered a cross-default provision under Section {section} following the breach of the senior secured credit facility covenants.",
    "Management disclosed that the company was not in compliance with the debt-to-equity ratio covenant as of December 31, reporting a ratio of {ratio}x against the maximum of {max_ratio}x.",
    "The company received a waiver from lenders after breaching the fixed charge coverage ratio covenant, which requires a minimum of {min_ratio}x.",
    "As reported in Note {note}, the company violated the tangible net worth covenant by ${amount} million, triggering acceleration rights under the credit agreement.",
    "The borrower failed to make the required principal payment of ${amount} million due under Section {section} of the term loan agreement.",
    "Following the covenant breach, the lender exercised its right to increase the applicable interest rate by {bps} basis points as specified in the default provisions.",
    "The company's failure to deliver audited financial statements within 90 days constitutes an event of default under Section {section} of the indenture.",
    "The restricted payments covenant under Section {section} was violated when the company declared dividends exceeding the permitted basket of ${amount} million.",
    "The company breached the capital expenditure limitation of ${amount} million specified in the credit agreement, with actual capex of ${amount2} million.",
    "An event of default occurred under the revolving credit facility due to the company's failure to maintain the required minimum liquidity of ${amount} million.",
    "The borrower's leverage ratio of {ratio}x exceeded the step-down threshold of {max_ratio}x required for the fiscal quarter ending March 31.",
    "The company failed to satisfy the debt incurrence test under Section {section}, which requires a pro forma fixed charge coverage ratio of at least {min_ratio}x.",
    "Lenders declared an event of default after the company's DSCR declined to {ratio}x, below the {min_ratio}x threshold for two consecutive quarters.",
    "The subsidiary guarantee covenant was breached when the company failed to cause its material subsidiaries representing {pct}% of consolidated assets to become guarantors.",
    "The company was in technical default of the maximum total debt to capitalization ratio of {max_ratio}%, with actual ratio at {ratio}%.",
    "The negative pledge covenant under Section {section} was violated by the granting of unauthorized liens on assets totaling ${amount} million.",
    "Due to the restatement of prior period financials, the company retroactively breached the minimum consolidated net income covenant for fiscal year {year}.",
    "The change of control provision under Section {section} was triggered by the acquisition, constituting an event of default requiring mandatory prepayment.",
    "The company failed to maintain the required debt service reserve fund balance of ${amount} million as stipulated in Section {section} of the bond indenture.",
]

# No-breach examples - language indicating compliance or general financial discussion
no_breach_templates = [
    # Compliant covenant language
    "As of December 31, the company was in compliance with all financial covenants under its credit facilities, with a DSCR of {ratio}x against the required {min_ratio}x.",
    "The company maintained its leverage ratio at {ratio}x, well within the maximum covenant level of {max_ratio}x specified in the revolving credit agreement.",
    "All financial covenants under the senior secured credit facility were satisfied as of the reporting date, with adequate headroom across all metrics.",
    "The interest coverage ratio of {ratio}x exceeded the minimum requirement of {min_ratio}x under Section {section} of the credit agreement.",
    "Management confirms that the company was in full compliance with all restrictive covenants throughout the fiscal year ended December 31.",
    "The company's current ratio of {ratio}x remained above the {min_ratio}x minimum threshold required under the revolving credit facility.",
    "Net worth as of the balance sheet date was ${amount} million, exceeding the minimum covenant requirement of ${min_amount} million by a comfortable margin.",
    # General financial statements (not covenant-related)
    "Total revenue for the fiscal year increased by {pct}% to ${amount} million, driven primarily by strong performance in the North American segment.",
    "Operating expenses decreased by {pct}% year-over-year due to the implementation of cost reduction initiatives across all business segments.",
    "The company reported net income of ${amount} million for the quarter, representing a {pct}% increase compared to the same period in the prior year.",
    "Cash and cash equivalents totaled ${amount} million at year end, compared to ${amount2} million in the prior year.",
    "Capital expenditures for the year were ${amount} million, primarily related to the expansion of manufacturing facilities in the Southeast region.",
    "The company completed the issuance of ${amount} million in senior unsecured notes bearing interest at {rate}% per annum, maturing in {year}.",
    "Depreciation and amortization expense for the period was ${amount} million, consistent with the prior year level.",
    "The effective tax rate for the fiscal year was {pct}%, compared to {pct2}% in the prior year, reflecting the impact of recent tax legislation.",
    "Research and development expenses increased to ${amount} million, representing {pct}% of total revenue for the fiscal year.",
    "The company's backlog at year end stood at ${amount} million, representing approximately {months} months of revenue at current run rates.",
    "Accounts receivable turnover improved to {ratio}x from {ratio2}x in the prior year, reflecting enhanced collection processes.",
    "The board of directors authorized a share repurchase program of up to ${amount} million over the next twelve months.",
    "Goodwill and intangible assets represented ${amount} million on the consolidated balance sheet as of December 31.",
    "The company entered into interest rate swap agreements with a notional value of ${amount} million to manage exposure to variable rate debt.",
    "Inventory levels decreased by {pct}% to ${amount} million due to improved supply chain management and demand forecasting.",
    "The pension plan was funded at {pct}% of the projected benefit obligation as of the measurement date.",
    "Foreign currency translation adjustments resulted in a ${amount} million decrease in accumulated other comprehensive income.",
    "The company's weighted average cost of debt was {rate}% for the fiscal year, compared to {rate2}% in the prior period.",
]

def fill_template(template):
    """Fill placeholders with realistic random values"""
    replacements = {
        '{section}': random.choice(['4.1', '4.2', '5.1', '5.3', '6.1', '6.2', '7.1', '7.2', '8.1', '9.1']),
        '{ratio}': f"{random.uniform(0.5, 8.0):.2f}",
        '{ratio2}': f"{random.uniform(0.5, 8.0):.2f}",
        '{min_ratio}': f"{random.uniform(1.0, 3.0):.2f}",
        '{max_ratio}': f"{random.uniform(3.0, 7.0):.2f}",
        '{amount}': str(random.choice([10, 15, 25, 50, 75, 100, 150, 200, 250, 500, 750])),
        '{amount2}': str(random.choice([20, 30, 50, 75, 100, 150, 200, 300, 400, 600])),
        '{min_amount}': str(random.choice([50, 75, 100, 150, 200])),
        '{bps}': str(random.choice([25, 50, 75, 100, 150, 200])),
        '{pct}': str(random.choice([5, 8, 10, 12, 15, 18, 20, 25, 30, 85, 90, 95])),
        '{pct2}': str(random.choice([5, 8, 10, 12, 15, 18, 20, 22, 25, 28])),
        '{rate}': f"{random.uniform(3.0, 8.0):.2f}",
        '{rate2}': f"{random.uniform(3.0, 8.0):.2f}",
        '{year}': str(random.choice([2023, 2024, 2025, 2026, 2027, 2028])),
        '{note}': str(random.choice([5, 6, 7, 8, 9, 10, 11, 12, 14, 15])),
        '{months}': str(random.choice([6, 8, 10, 12, 14, 18])),
    }
    result = template
    for key, value in replacements.items():
        result = result.replace(key, value, 1)
    return result

# Generate training data: 500 examples of each class (1000 total)
# More examples per template = more variety in the numeric values
breach_examples = []
for _ in range(500):
    template = random.choice(breach_templates)
    breach_examples.append({"text": fill_template(template), "label": 1})

no_breach_examples = []
for _ in range(500):
    template = random.choice(no_breach_templates)
    no_breach_examples.append({"text": fill_template(template), "label": 0})

all_examples = breach_examples + no_breach_examples
random.shuffle(all_examples)

print(f"Total examples: {len(all_examples)}")
print(f"Breach examples: {len(breach_examples)}")
print(f"No-breach examples: {len(no_breach_examples)}")
print(f"\n--- Sample BREACH example ---")
print(f"[Label: {breach_examples[0]['label']}] {breach_examples[0]['text'][:150]}...")
print(f"\n--- Sample NO-BREACH example ---")
print(f"[Label: {no_breach_examples[0]['label']}] {no_breach_examples[0]['text'][:150]}...")

Total examples: 1000
Breach examples: 500
No-breach examples: 500

--- Sample BREACH example ---
[Label: 1] The company was in technical default of the maximum total debt to capitalization ratio of 5.95%, with actual ratio at 0.69%....

--- Sample NO-BREACH example ---
[Label: 0] Foreign currency translation adjustments resulted in a $750 million decrease in accumulated other comprehensive income....


In [4]:
# Cell 3: Convert to HuggingFace Dataset and split into train/validation
from datasets import Dataset, DatasetDict, ClassLabel
from collections import Counter

# Create HuggingFace Dataset from our list of dicts
full_dataset = Dataset.from_list(all_examples)

# Cast the label column to ClassLabel type (required for stratified split)
full_dataset = full_dataset.cast_column('label', ClassLabel(names=['no_breach', 'breach']))

# 85% train, 15% validation (stratified by label)
split = full_dataset.train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column='label'
)

dataset_split = DatasetDict({
    'train': split['train'],
    'validation': split['test']
})

print(f"Training samples: {len(dataset_split['train'])}")
print(f"Validation samples: {len(dataset_split['validation'])}")

label_map = {0: "no_breach", 1: "breach"}
for split_name in ['train', 'validation']:
    labels = [ex['label'] for ex in dataset_split[split_name]]
    counts = Counter(labels)
    total = len(labels)
    print(f"\n{split_name} distribution:")
    for label_id in sorted(counts.keys()):
        print(f"  {label_map[label_id]}: {counts[label_id]} ({counts[label_id]/total*100:.1f}%)")

Casting the dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training samples: 850
Validation samples: 150

train distribution:
  no_breach: 425 (50.0%)
  breach: 425 (50.0%)

validation distribution:
  no_breach: 75 (50.0%)
  breach: 75 (50.0%)


In [5]:
# Cell 4: Load FinBERT tokenizer and tokenize the dataset
from transformers import AutoTokenizer

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer loaded: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_dataset = dataset_split.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing"
)

# Remove raw text column, rename label column
tokenized_dataset = tokenized_dataset.remove_columns(['text'])
tokenized_dataset = tokenized_dataset.rename_column('label', 'labels')
tokenized_dataset.set_format('torch')

print(f"\nTokenized dataset ready!")
print(f"Features: {tokenized_dataset['train'].column_names}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: ProsusAI/finbert
Vocabulary size: 30522


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Tokenizing:   0%|          | 0/850 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/150 [00:00<?, ? examples/s]


Tokenized dataset ready!
Features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [7]:
# Cell 5: Load pre-trained FinBERT with 2-class classification head
import torch
from transformers import AutoModelForSequenceClassification

# num_labels=2: breach (1) or no-breach (0)
# ignore_mismatched_sizes=True because FinBERT's original checkpoint has a 3-class
# classifier head (positive/neutral/negative) but we need 2 classes.
# This discards the old classifier weights and initializes a fresh 2-class head.
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model loaded on: {device}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 109,483,778
Trainable parameters: 109,483,778
Model loaded on: cuda


In [8]:
# Cell 6: Define evaluation metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='binary'),
        'precision': precision_score(labels, predictions, average='binary'),
        'recall': recall_score(labels, predictions, average='binary'),
    }

print("✅ Metrics function defined!")

✅ Metrics function defined!


In [9]:
# Cell 7: Configure and run training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./finbert-breach-training',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=25,
    fp16=True,
    seed=42,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

print("🚀 Starting FinBERT breach detection fine-tuning...")
print(f"   Training samples: {len(tokenized_dataset['train'])}")
print(f"   Validation samples: {len(tokenized_dataset['validation'])}")
print(f"   Epochs: 5")
print(f"   This will take approximately 10-15 minutes on T4 GPU\n")

train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Total training time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"   Final training loss: {train_result.metrics['train_loss']:.4f}")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🚀 Starting FinBERT breach detection fine-tuning...
   Training samples: 850
   Validation samples: 150
   Epochs: 5
   This will take approximately 10-15 minutes on T4 GPU



Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.098200,0.016332,1.000000,1.000000,1.000000,1.000000
2,0.003400,0.001639,1.000000,1.000000,1.000000,1.000000
3,0.001600,0.000954,1.000000,1.000000,1.000000,1.000000
4,0.001100,0.000731,1.000000,1.000000,1.000000,1.000000
5,0.001000,0.000668,1.000000,1.000000,1.000000,1.000000



✅ Training complete!
   Total training time: 173 seconds
   Final training loss: 0.0603


In [10]:
# Cell 8: Detailed evaluation
eval_results = trainer.evaluate()

print("=" * 60)
print("FINBERT BREACH DETECTION — FINAL EVALUATION RESULTS")
print("=" * 60)
for key, value in eval_results.items():
    if key.startswith('eval_'):
        clean_key = key.replace('eval_', '')
        if isinstance(value, float):
            print(f"  {clean_key:>25s}: {value:.4f}")
        else:
            print(f"  {clean_key:>25s}: {value}")

# Detailed classification report
print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)

predictions_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

target_names = ['no_breach', 'breach']
report = classification_report(true_labels, predictions, target_names=target_names, digits=4)
print(report)

FINBERT BREACH DETECTION — FINAL EVALUATION RESULTS
                       loss: 0.0163
                   accuracy: 1.0000
                         f1: 1.0000
                  precision: 1.0000
                     recall: 1.0000
                    runtime: 1.2263
         samples_per_second: 122.3230
           steps_per_second: 4.0770

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

   no_breach     1.0000    1.0000    1.0000        75
      breach     1.0000    1.0000    1.0000        75

    accuracy                         1.0000       150
   macro avg     1.0000    1.0000    1.0000       150
weighted avg     1.0000    1.0000    1.0000       150



In [11]:
# Cell 9: Save model to Google Drive
from google.colab import drive
import os

# Mount Google Drive (may already be mounted from previous notebook)
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/finsight-ai-models/finbert-breach'
os.makedirs(save_dir, exist_ok=True)

# Save model and tokenizer
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("✅ Model saved to Google Drive!")
print(f"   Location: {save_dir}")
print(f"\n   Saved files:")
for f in sorted(os.listdir(save_dir)):
    size_mb = os.path.getsize(os.path.join(save_dir, f)) / 1e6
    print(f"   {f:40s} ({size_mb:.1f} MB)")

Mounted at /content/drive
✅ Model saved to Google Drive!
   Location: /content/drive/MyDrive/finsight-ai-models/finbert-breach

   Saved files:
   config.json                              (0.0 MB)
   model.safetensors                        (438.0 MB)
   special_tokens_map.json                  (0.0 MB)
   tokenizer.json                           (0.7 MB)
   tokenizer_config.json                    (0.0 MB)
   training_args.bin                        (0.0 MB)
   vocab.txt                                (0.2 MB)


In [12]:
# Cell 10: Save training metrics for documentation
import json

metrics_to_save = {
    "model": "ProsusAI/finbert",
    "task": "covenant_breach_detection",
    "num_classes": 2,
    "class_names": ["no_breach", "breach"],
    "dataset": "synthetic_covenant_clauses",
    "dataset_note": "1000 synthetic examples generated from 25 breach and 25 no-breach templates with randomized financial values, mimicking real SEC EDGAR 10-K covenant language",
    "train_samples": len(tokenized_dataset['train']),
    "val_samples": len(tokenized_dataset['validation']),
    "epochs": 5,
    "learning_rate": 2e-5,
    "batch_size": 16,
    "eval_results": {k.replace('eval_', ''): round(v, 4) if isinstance(v, float) else v
                     for k, v in eval_results.items()},
    "training_time_seconds": round(train_result.metrics['train_runtime'], 0),
}

metrics_path = '/content/drive/MyDrive/finsight-ai-models/finbert-breach/training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_to_save, f, indent=2)

print("✅ Training metrics saved!")
print(json.dumps(metrics_to_save, indent=2))

✅ Training metrics saved!
{
  "model": "ProsusAI/finbert",
  "task": "covenant_breach_detection",
  "num_classes": 2,
  "class_names": [
    "no_breach",
    "breach"
  ],
  "dataset": "synthetic_covenant_clauses",
  "dataset_note": "1000 synthetic examples generated from 25 breach and 25 no-breach templates with randomized financial values, mimicking real SEC EDGAR 10-K covenant language",
  "train_samples": 850,
  "val_samples": 150,
  "epochs": 5,
  "learning_rate": 2e-05,
  "batch_size": 16,
  "eval_results": {
    "loss": 0.0163,
    "accuracy": 1.0,
    "f1": 1.0,
    "precision": 1.0,
    "recall": 1.0,
    "runtime": 1.2263,
    "samples_per_second": 122.323,
    "steps_per_second": 4.077,
    "epoch": 5.0
  },
  "training_time_seconds": 173.0
}
